# M01E · Enrichment: Pairwise Sequence Alignment & Dynamic Programming (Solutions)
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M01_Python_Basics/solutions/M01E_Enrichment_Sequence_Alignment_Solutions.ipynb)

⏱ ~90 min · Difficulty: ★★★ · Prerequisites: M01A, M01B · **Status: Optional / Enrichment (Not Examined)**


## 1. Learning Outcomes

By the end of this enrichment notebook, you will be able to:
1. **Explain** the evolutionary rationale for pairwise sequence alignment (homology, substitution events, and insertions/deletions).
2. **Differentiate** between ungapped distance metrics (Hamming distance) and gapped alignment scoring schemes.
3. **Formulate and initialize** the dynamic programming scoring matrix for global alignment (Needleman-Wunsch).
4. **Implement** the dynamic programming recurrence relation and traceback algorithm from scratch in pure Python.
5. **Reconstruct** optimal aligned sequences and calculate percentage sequence identity and gap frequency.
6. **Validate** your pure Python implementation by demonstrating parity against Biopython's `Bio.Align.PairwiseAligner`.
7. **Evaluate** affine gap penalties and construct 2D comparative sequence dot plots.


## 2. Prerequisites

- Completion of **M01A (Core Python)** and **M01B (Sequence Manipulation)**.
- Familiarity with 2D lists (matrices), nested loops, and basic recursion / traceback principles.


## 3. Setup

Run this cell to detect the runtime environment, configure paths, and import dependencies.


In [ ]:
# --- Setup -------------------------------------------------------
import sys
import os
import platform
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Installing Biopython...")
    !pip -q install biopython
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    p = Path.cwd()
    found_root = None
    for candidate in [p] + list(p.parents):
        if (candidate / "data" / "sequences").exists() or (candidate / "COURSE_PLAN.md").exists():
            found_root = candidate
            break
    if found_root:
        ROOT = found_root
    else:
        repo_candidate = Path("/Users/prateekpaul/Library/CloudStorage/GoogleDrive-prateekp@iiitd.ac.in/My Drive/IIITD/mygithub/ProgrammingCB")
        ROOT = repo_candidate if repo_candidate.exists() else p

DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / "M01E"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Python version     : {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"✓ Repository Root    : {ROOT.resolve()}")
print(f"✓ Data Directory     : {DATA_DIR.resolve()}")
print(f"✓ Output Directory   : {OUT_DIR.resolve()}")

try:
    from Bio.Align import PairwiseAligner
    print("✓ Biopython PairwiseAligner loaded successfully.")
except ImportError:
    print("⚠️ Biopython not installed. Local comparison will be skipped.")


## 4. Concept Sections

---

### 1E.1 Biological Rationale & Evolutionary Alignment Models

#### 4.1.1 Theory: Sequence Similarity vs. Homology
In computational biology, sequence alignment is the fundamental method used to infer **homology**—common evolutionary ancestry.
Mutations include point substitutions and insertions/deletions (indels leading to alignment gaps).


---

### 1E.2 Hamming Distance vs. Gapped Alignment

#### 4.2.1 Theory: The Limitation of Ungapped Distance
Hamming distance counts mismatch positions in equal-length strings, but fails if an indel shifts the reading frame.


In [ ]:
def hamming_distance(s1: str, s2: str) -> int:
    if len(s1) != len(s2): raise ValueError("Equal length required!")
    return sum(1 for c1, c2 in zip(s1.upper(), s2.upper()) if c1 != c2)

def score_ungapped_alignment(s1: str, s2: str, match: int = 1, mismatch: int = -1) -> int:
    return sum(match if c1 == c2 else mismatch for c1, c2 in zip(s1.upper(), s2.upper()))

h_dist = hamming_distance("GATTACA", "GCATGCA")
aln_score = score_ungapped_alignment("GATTACA", "GCATGCA")
assert h_dist == 3
assert aln_score == 1


---

### 1E.3 The Needleman-Wunsch Global Alignment Algorithm

#### 4.3.1 Theory: Dynamic Programming Matrix Formulation
Needleman-Wunsch (1970) guarantees optimal global alignment across entire sequence lengths.
Base cases: $M(i, 0) = i \times gap, M(0, j) = j \times gap$.
Recurrence: $\max(diag, up, left)$.


In [ ]:
def initialize_nw_matrix(len1: int, len2: int, gap_penalty: int = -2) -> list[list[int]]:
    matrix = [[0] * (len2 + 1) for _ in range(len1 + 1)]
    for i in range(len1 + 1): matrix[i][0] = i * gap_penalty
    for j in range(len2 + 1): matrix[0][j] = j * gap_penalty
    return matrix

def fill_nw_matrix(s1: str, s2: str, match: int = 1, mismatch: int = -1, gap: int = -2):
    m, n = len(s1), len(s2)
    matrix = initialize_nw_matrix(m, n, gap)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            char1 = s1[i-1]
            char2 = s2[j-1]
            diag = matrix[i-1][j-1] + (match if char1 == char2 else mismatch)
            up = matrix[i-1][j] + gap
            left = matrix[i][j-1] + gap
            matrix[i][j] = max(diag, up, left)
    return matrix

nw_matrix = fill_nw_matrix("GATTACA", "GCATGCA", match=1, mismatch=-1, gap=-2)
assert nw_matrix[7][7] == 1


---

### 1E.4 Traceback & Sequence Reconstruction

#### 4.4.1 Theory: Reconstructing the Optimal Path
Traceback moves from $(m, n)$ back to $(0, 0)$ identifying which predecessor generated the maximal score.


In [ ]:
def needleman_wunsch(seq1: str, seq2: str, match: int = 1, mismatch: int = -1, gap: int = -2) -> tuple[int, str, str]:
    s1 = seq1.upper()
    s2 = seq2.upper()
    m, n = len(s1), len(s2)
    matrix = fill_nw_matrix(s1, s2, match, mismatch, gap)
    optimal_score = matrix[m][n]
    aln1, aln2 = [], []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            char_score = match if s1[i-1] == s2[j-1] else mismatch
            if matrix[i][j] == matrix[i-1][j-1] + char_score:
                aln1.append(s1[i-1])
                aln2.append(s2[j-1])
                i -= 1; j -= 1
                continue
        if i > 0 and matrix[i][j] == matrix[i-1][j] + gap:
            aln1.append(s1[i-1]); aln2.append("-")
            i -= 1
            continue
        if j > 0 and matrix[i][j] == matrix[i][j-1] + gap:
            aln1.append("-"); aln2.append(s2[j-1])
            j -= 1
            continue
    return (optimal_score, "".join(reversed(aln1)), "".join(reversed(aln2)))

score, a1, a2 = needleman_wunsch("GATTACA", "GCATGCA")
assert score == 1


---

### 1E.5 Validation & Parity with Biopython's PairwiseAligner

#### 4.5.1 Industry Standard Parity
Compare pure Python implementation with Biopython's `Bio.Align.PairwiseAligner`.


In [ ]:
try:
    from Bio.Align import PairwiseAligner
    aligner = PairwiseAligner()
    aligner.mode = 'global'
    aligner.match_score = 1.0
    aligner.mismatch_score = -1.0
    aligner.open_gap_score = -2.0
    aligner.extend_gap_score = -2.0
    bio_score = aligner.score("GATTACA", "GCATGCA")
    assert score == bio_score
    print("✓ Full parity with Biopython verified!")
except ImportError:
    pass


## 5. Practice Exercises

Implement each of the 7 alignment utilities below.


### Exercise 1 (Level 1 ★): Hamming Distance Validator

**Problem**: Write a function `validate_and_hamming(s1: str, s2: str) -> int` validating equal lengths and canonical nucleotides before computing mismatch count.

- **Input**: Two strings `s1` and `s2`.
- **Expected Output**: Integer mismatch count.


In [ ]:
def validate_and_hamming(s1: str, s2: str) -> int:
    if len(s1) != len(s2): raise ValueError("Unequal lengths!")
    c1, c2 = s1.upper(), s2.upper()
    valid = set("ACGT")
    if any(b not in valid for b in c1) or any(b not in valid for b in c2):
        raise ValueError("Invalid bases!")
    return sum(1 for a, b in zip(c1, c2) if a != b)


#### 💡 Solution Explanation
Validates alphabet and computes point mismatch count.

In [ ]:
assert validate_and_hamming('ATGC', 'ACGC') == 1
print('✓ Exercise 1 passed!')


### Exercise 2 (Level 1 ★): Alignment Statistics Calculator

**Problem**: Write a function `calculate_alignment_stats(aln1: str, aln2: str) -> dict` returning `{'identity_pct', 'gap_pct', 'length'}`.

- **Input**: Two aligned equal-length strings.
- **Expected Output**: Statistics dictionary.


In [ ]:
def calculate_alignment_stats(aln1: str, aln2: str) -> dict:
    if len(aln1) != len(aln2) or len(aln1) == 0: raise ValueError("Invalid alignment!")
    L = len(aln1)
    m = sum(1 for c1, c2 in zip(aln1.upper(), aln2.upper()) if c1 == c2 and c1 != "-")
    g = sum(1 for c1, c2 in zip(aln1, aln2) if c1 == "-" or c2 == "-")
    return {"identity_pct": round((m / L) * 100.0, 2), "gap_pct": round((g / L) * 100.0, 2), "length": L}


#### 💡 Solution Explanation
Tallies matches and gaps across alignment columns.

In [ ]:
stats = calculate_alignment_stats('GATT-ACA', 'G-TTGACA')
assert stats['identity_pct'] == round((6 / 8) * 100.0, 2)
print('✓ Exercise 2 passed!')


### Exercise 3 (Level 2 ★★): Needleman-Wunsch with Ts/Tv Substitution Scoring

**Problem**: Write a function `needleman_wunsch_tstv_score(seq1: str, seq2: str, match: int = 2, transition: int = -1, transversion: int = -2, gap: int = -2) -> int` incorporating transition/transversion differentiation.

- **Input**: Two DNA strings and scores.
- **Expected Output**: Optimal global alignment score.


In [ ]:
def needleman_wunsch_tstv_score(seq1: str, seq2: str, match: int = 2, transition: int = -1, transversion: int = -2, gap: int = -2) -> int:
    s1, s2 = seq1.upper(), seq2.upper()
    m, n = len(s1), len(s2)
    pur, pyr = {"A", "G"}, {"C", "T"}
    def sub(b1, b2):
        if b1 == b2: return match
        if (b1 in pur and b2 in pur) or (b1 in pyr and b2 in pyr): return transition
        return transversion
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i * gap
    for j in range(n + 1): dp[0][j] = j * gap
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            diag = dp[i-1][j-1] + sub(s1[i-1], s2[j-1])
            up = dp[i-1][j] + gap
            left = dp[i][j-1] + gap
            dp[i][j] = max(diag, up, left)
    return dp[m][n]


#### 💡 Solution Explanation
Dynamic programming recurrence with substitution matrix reflecting biochemical transition/transversion rates.

In [ ]:
assert needleman_wunsch_tstv_score('A', 'G') == -1
assert needleman_wunsch_tstv_score('A', 'C') == -2
print('✓ Exercise 3 passed!')


### Exercise 4 (Level 3 ★★★): Smith-Waterman Local Alignment Peak Finder

**Problem**: Write a function `smith_waterman_peak(seq1: str, seq2: str, match: int = 2, mismatch: int = -1, gap: int = -2) -> tuple[int, int, int]` resetting negative values to 0 and returning `(peak_score, peak_row, peak_col)`.

- **Input**: Two DNA strings and scores.
- **Expected Output**: Tuple with peak score and coordinates.


In [ ]:
def smith_waterman_peak(seq1: str, seq2: str, match: int = 2, mismatch: int = -1, gap: int = -2) -> tuple[int, int, int]:
    s1, s2 = seq1.upper(), seq2.upper()
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    max_s, max_i, max_j = 0, 0, 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            score = match if s1[i-1] == s2[j-1] else mismatch
            v = max(0, dp[i-1][j-1] + score, dp[i-1][j] + gap, dp[i][j-1] + gap)
            dp[i][j] = v
            if v > max_s: max_s, max_i, max_j = v, i, j
    return (max_s, max_i, max_j)


#### 💡 Solution Explanation
Local alignment non-negativity constraint.

In [ ]:
p, pi, pj = smith_waterman_peak('AAATACGTTT', 'GGGTACGGGG')
assert p == 8
print('✓ Exercise 4 passed!')


### Exercise 5 (Level 2 ★★): Affine Gap Penalty Alignment Scorer

**Problem**: Write a function `score_affine_alignment(aln1: str, aln2: str, match: int = 1, mismatch: int = -1, gap_open: int = -3, gap_extend: int = -1) -> int` scoring pre-aligned sequences under affine gap costs: contiguous runs of gaps charge `gap_open` for the first gap and `gap_extend` for subsequent gaps in the run.

- **Input**: Two aligned strings and scoring parameters.
- **Expected Output**: Integer score.


In [ ]:
def score_affine_alignment(aln1: str, aln2: str, match: int = 1, mismatch: int = -1, gap_open: int = -3, gap_extend: int = -1) -> int:
    score = 0
    in_gap_1 = False
    in_gap_2 = False
    for c1, c2 in zip(aln1.upper(), aln2.upper()):
        if c1 != "-" and c2 != "-":
            in_gap_1 = False
            in_gap_2 = False
            score += match if c1 == c2 else mismatch
        elif c1 == "-":
            in_gap_2 = False
            score += gap_extend if in_gap_1 else gap_open
            in_gap_1 = True
        elif c2 == "-":
            in_gap_1 = False
            score += gap_extend if in_gap_2 else gap_open
            in_gap_2 = True
    return score


#### 💡 Solution Explanation
Biologically, a single multi-base deletion is much more likely than multiple independent 1-base deletions, which is modeled by affine gap penalties ($W = g_{open} + (k-1) \times g_{extend}$).

In [ ]:
# Match 3 bp (+3), a 3-bp gap in seq2 (gap_open + 2*gap_extend = -3 - 2 = -5) -> total = -2
a1 = "ATG---"
a2 = "ATGCCC"
assert score_affine_alignment(a1, a2, match=1, mismatch=-1, gap_open=-3, gap_extend=-1) == -2
print('✓ Exercise 5 passed!')


### Exercise 6 (Level 2 ★★): 2D Sequence Dot Plot Matrix Generator

**Problem**: Write a function `generate_dotplot(seq1: str, seq2: str, window: int = 3, threshold: int = 3) -> list[list[bool]]` creating an $M \times N$ boolean matrix where `matrix[i][j]` is `True` if the substring of length `window` starting at `seq1[i]` matches `seq2[j]` at $\ge$ `threshold` positions.

- **Input**: Two DNA strings and integers `window`, `threshold`.
- **Expected Output**: 2D list of booleans.


In [ ]:
def generate_dotplot(seq1: str, seq2: str, window: int = 3, threshold: int = 3) -> list[list[bool]]:
    s1, s2 = seq1.upper(), seq2.upper()
    rows = len(s1) - window + 1
    cols = len(s2) - window + 1
    if rows <= 0 or cols <= 0: return []
    plot = [[False] * cols for _ in range(rows)]
    for i in range(rows):
        w1 = s1[i:i+window]
        for j in range(cols):
            w2 = s2[j:j+window]
            matches = sum(1 for a, b in zip(w1, w2) if a == b)
            if matches >= threshold:
                plot[i][j] = True
    return plot


#### 💡 Solution Explanation
Dot plots visualize sequence similarity, direct repeats, and genomic inversions as diagonals across a 2D grid.

In [ ]:
dp = generate_dotplot("ATGC", "ATGC", window=2, threshold=2)
# Diagonals must match: (0,0), (1,1), (2,2)
assert dp[0][0] is True and dp[1][1] is True and dp[2][2] is True
assert dp[0][1] is False
print('✓ Exercise 6 passed!')


### Exercise 7 (Level 3 ★★★): Smith-Waterman Local Alignment Full Traceback

**Problem**: Write a function `smith_waterman_align(seq1: str, seq2: str, match: int = 2, mismatch: int = -1, gap: int = -2) -> tuple[int, str, str]` that finds the optimal local alignment score, locates the peak cell, traces back until a cell with score 0 is reached, and returns `(peak_score, aligned_s1_sub, aligned_s2_sub)`.

- **Input**: Two DNA strings and scores.
- **Expected Output**: Tuple with peak score and local aligned strings.


In [ ]:
def smith_waterman_align(seq1: str, seq2: str, match: int = 2, mismatch: int = -1, gap: int = -2) -> tuple[int, str, str]:
    s1, s2 = seq1.upper(), seq2.upper()
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    max_s, max_i, max_j = 0, 0, 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            score = match if s1[i-1] == s2[j-1] else mismatch
            v = max(0, dp[i-1][j-1] + score, dp[i-1][j] + gap, dp[i][j-1] + gap)
            dp[i][j] = v
            if v > max_s: max_s, max_i, max_j = v, i, j
    aln1, aln2 = [], []
    i, j = max_i, max_j
    while i > 0 and j > 0 and dp[i][j] > 0:
        score = match if s1[i-1] == s2[j-1] else mismatch
        if dp[i][j] == dp[i-1][j-1] + score:
            aln1.append(s1[i-1]); aln2.append(s2[j-1])
            i -= 1; j -= 1
        elif dp[i][j] == dp[i-1][j] + gap:
            aln1.append(s1[i-1]); aln2.append("-")
            i -= 1
        elif dp[i][j] == dp[i][j-1] + gap:
            aln1.append("-"); aln2.append(s2[j-1])
            j -= 1
        else:
            break
    return (max_s, "".join(reversed(aln1)), "".join(reversed(aln2)))


#### 💡 Solution Explanation
Completes the Smith-Waterman local alignment by tracing back from the highest-scoring matrix cell until a score of zero terminates the local alignment.

In [ ]:
score, a1, a2 = smith_waterman_align('NNTACGNN', 'ZZTACGZZ')
assert score == 8
assert a1 == 'TACG'
assert a2 == 'TACG'
print('✓ Exercise 7 passed!')


## 6. Summary & Cheat Sheet Table

| Alignment Type | Algorithm | Time Complexity | Recurrence / Initialization | Use Case |
|---|---|---|---|---|
| **Hamming Distance** | Direct character zip | $O(N)$ | $d = \sum [s_1[i] \ne s_2[i]]$ | Equal-length un-gapped comparison |
| **Global Alignment** | Needleman-Wunsch (1970) | $O(M \times N)$ | $M(0, j) = j \cdot \text{gap}$, $\max(\text{diag}, \text{up}, \text{left})$ | Full-length homologous genes/proteins |
| **Local Alignment** | Smith-Waterman (1981) | $O(M \times N)$ | $M(0, j) = 0$, $\max(0, \text{diag}, \text{up}, \text{left})$ | Conserved domains, local motifs |
| **Affine Gap Score** | Gotoh (1982) | $O(M \times N)$ | $W = g_{open} + (k-1)g_{extend}$ | Natural indel biology |
| **Dot Plot** | Sliding window match | $O(M \times N)$ | Boolean grid matching | Macro-synteny, inversions, repeats |
| **Biopython** | `Bio.Align.PairwiseAligner` | $O(M \times N)$ (C-optimized) | Configurable scoring & affine gaps | Production genomics pipelines |


## 7. Further Reading & References

1. **Needleman, S. B., & Wunsch, C. D. (1970)**. A general method applicable to the search for similarities in the amino acid sequence of two proteins. *Journal of Molecular Biology*, 48(3), 443-453.
2. **Smith, T. F., & Waterman, M. S. (1981)**. Identification of common molecular subsequences. *Journal of Molecular Biology*, 147(1), 195-197.
3. **Gotoh, O. (1982)**. An improved algorithm for matching biological sequences. *Journal of Molecular Biology*, 162(3), 705-708. (The affine gap penalty algorithm).
4. **Biopython Align Tutorial**: https://biopython.org/docs/latest/api/Bio.Align.html
